# 03 — Model comparison

Three tiers of complexity. The question is not which has the highest AUC — it is whether the extra complexity changes the **policy**. That gets answered in 04.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.config import load_config

cfg = load_config()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

import pickle
from src.splits import assign_split
from src.data_load import months_of_credit_history

df = pd.read_parquet('../data/processed/modelling_sample.parquet')
df['credit_history_months'] = months_of_credit_history(df['earliest_cr_line'], df['issue_dt'])
df['split'] = assign_split(df, cfg)
tr, va = df[df['split']=='train'], df[df['split']=='valid']
with open('../data/processed/fitted.pkl','rb') as f:
    fit = pickle.load(f)
sc, gbm, NUM = fit['scorecard'], fit['gbm'], fit['NUM']

## 1. Tier 1 — a rule any analyst could build in an afternoon

If a boosted model on 14 features cannot beat FICO + DTI on the policy curve, that is the finding.

In [ ]:
from src.models import simple_rule_score
from src.metrics import summary_metrics

summary_metrics(va['target'].to_numpy(), simple_rule_score(va))

## 2–4. All tiers, plus the benchmark and the stack

Note `accuracy` is absent throughout. At a 14% bad rate, 'approve everyone' scores 86% accurate and approves every defaulter.

In [ ]:
from src.policy import grade_to_score

scores = {
    'tier1_rule':          simple_rule_score(va),
    'tier2_scorecard':     sc.predict_proba(va),
    'tier3_gbm':           gbm.predict_proba(va[NUM])[:,1],
    'platform_sub_grade':  grade_to_score(va['sub_grade']).to_numpy(),
}
# The lender already HAS the grade, so stacking is available even though
# grade is excluded from the feature set.
scores['stacked'] = (pd.Series(scores['platform_sub_grade']).rank(pct=True).to_numpy()
                     + pd.Series(scores['tier2_scorecard']).rank(pct=True).to_numpy()) / 2

pd.DataFrame([{**summary_metrics(va['target'].to_numpy(), s), 'model': n}
              for n, s in scores.items()])[
    ['model','auc','pr_auc','ks','capture_top_10pct']]

**Verified result.** Neither the scorecard (AUC 0.662) nor the GBM (0.663) beats `sub_grade` (0.675) from application-time data alone. The platform had inputs this data does not expose: the full credit report, its own verification, and visibility into the applicants it declined.

Stacking the scorecard *on top of* the grade reaches 0.687. The scorecard separates risk **within** a grade — it finds the safer B3s and the riskier B3s. Incremental lift over the incumbent is also the question a risk team actually asks.